# Task 3 — MongoDB and PostgreSQL

This notebook covers:
1. Loading raw Slack data into a local MongoDB instance
2. Designing a PostgreSQL schema to store ML features
3. Creating the tables using Python
4. Loading ML features into PostgreSQL

## 0. Setup

In [3]:
import sys
sys.path.insert(0, '..')

import json
import os
import re
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import pymongo
import psycopg2
from psycopg2.extras import execute_values

from src.loader import SlackDataLoader
from src.utils import add_time_columns

# Load all Slack data
loader = SlackDataLoader()
df_raw = loader.load_all_channels()
df = add_time_columns(df_raw.copy())
df = df[df['type'] == 'message'].copy()
df = df[df['text'].notna() & (df['text'].str.strip() != '')].copy()

print(f'Total messages loaded: {len(df):,}')
print(f'Channels: {df["channel"].nunique()}')

Total messages loaded: 19,602
Channels: 39


## 1. MongoDB — Load Raw Slack Data

### Schema Design

We split the Slack data into **4 collections** (like tables in SQL):

| Collection | What it stores |
|---|---|
| `messages` | All top-level messages, one document per message |
| `replies` | All thread replies, linked back to their parent message |
| `reactions` | All emoji reactions, linked to their message |
| `mentions` | All `<@USER>` mentions extracted from message text |

**Why split them?**
- Faster queries — searching only messages doesn't load reply data
- Efficient streaming — new replies/reactions can be inserted without touching the message
- Future-proof — works for any Slack workspace, not just Gebeya Batch 6

In [ ]:
# Connect to MongoDB
client = pymongo.MongoClient('mongodb://localhost:27017/')
db = client['gebeya_slack']

# Drop existing collections so we start fresh each run
db.messages.drop()
db.replies.drop()
db.reactions.drop()
db.mentions.drop()

print('Connected to MongoDB database: gebeya_slack')
print('Collections cleared and ready')

In [ ]:
messages_docs = []
replies_docs = []
reactions_docs = []
mentions_docs = []

MENTION_RE = re.compile(r'<@([A-Za-z0-9]+)>')

for _, row in df.iterrows():
    is_reply = (
        'thread_ts' in row and
        pd.notna(row.get('thread_ts')) and
        row.get('thread_ts') != row.get('ts')
    )

    doc = {
        'message_id': str(row.get('ts', '')),
        'channel': row.get('channel', ''),
        'user': str(row.get('user', '')),
        'text': str(row.get('text', '')),
        'ts': str(row.get('ts', '')),
        'datetime': row.get('datetime').isoformat() if pd.notna(row.get('datetime')) else None,
        'thread_ts': str(row.get('thread_ts', '')) if pd.notna(row.get('thread_ts')) else None,
        'reply_count': int(row['reply_count']) if 'reply_count' in row and pd.notna(row.get('reply_count')) else 0,
        'reply_users_count': int(row['reply_users_count']) if 'reply_users_count' in row and pd.notna(row.get('reply_users_count')) else 0,
    }

    if is_reply:
        doc['parent_message_id'] = str(row.get('thread_ts', ''))
        replies_docs.append(doc)
    else:
        messages_docs.append(doc)

    # Extract reactions
    if isinstance(row.get('reactions'), list):
        for rxn in row['reactions']:
            reactions_docs.append({
                'message_id': str(row.get('ts', '')),
                'channel': row.get('channel', ''),
                'emoji': rxn.get('name', ''),
                'count': int(rxn.get('count', 1)),
                'users': rxn.get('users', []),
            })

    # Extract mentions (<@USERID> patterns in message text)
    text = str(row.get('text', ''))
    for mentioned_user in MENTION_RE.findall(text):
        mentions_docs.append({
            'message_id': str(row.get('ts', '')),
            'channel': row.get('channel', ''),
            'mentioned_by': str(row.get('user', '')),
            'mentioned_user': mentioned_user,
            'datetime': row.get('datetime').isoformat() if pd.notna(row.get('datetime')) else None,
        })

# Insert into MongoDB
if messages_docs:
    db.messages.insert_many(messages_docs)
if replies_docs:
    db.replies.insert_many(replies_docs)
if reactions_docs:
    db.reactions.insert_many(reactions_docs)
if mentions_docs:
    db.mentions.insert_many(mentions_docs)

print(f'Inserted {db.messages.count_documents({}):,} messages')
print(f'Inserted {db.replies.count_documents({}):,} replies')
print(f'Inserted {db.reactions.count_documents({}):,} reactions')
print(f'Inserted {db.mentions.count_documents({}):,} mentions')

In [ ]:
# Create indexes for fast querying
db.messages.create_index('channel')
db.messages.create_index('user')
db.messages.create_index('ts')
db.replies.create_index('parent_message_id')
db.replies.create_index('channel')
db.reactions.create_index('message_id')
db.mentions.create_index('mentioned_user')
db.mentions.create_index('mentioned_by')
db.mentions.create_index('channel')

print('Indexes created')

# Sample query — top 5 messages in one channel
sample = list(db.messages.find({'channel': 'all-week1'}).limit(3))
for s in sample:
    print(f"  user={s['user'][:8]}... text={s['text'][:60]}")

# Sample query — top 5 most mentioned users
print('\n--- Top 5 most mentioned users ---')
pipeline = [
    {'$group': {'_id': '$mentioned_user', 'count': {'$sum': 1}}},
    {'$sort': {'count': -1}},
    {'$limit': 5}
]
for doc in db.mentions.aggregate(pipeline):
    print(f"  {doc['_id']}: {doc['count']} mentions")

## 2. PostgreSQL — Feature Store

### Schema Design

We store the ML features computed in Task 1 & 2 into two tables:

**`user_features`** — one row per user, stores aggregated stats
```
user_id | channel | message_count | reply_count | reaction_count | mention_count | dominant_message_type
```

**`message_features`** — one row per message, stores per-message ML features
```
message_id | channel | user_id | text_length | reply_count | reaction_count | message_type | sentiment_score | hour_of_day | day_of_week
```

**`daily_sentiment`** — one row per day, stores sentiment over time
```
date | days_since_start | sentiment_score | message_count
```

In [5]:
# Connect to PostgreSQL
conn = psycopg2.connect(dbname='postgres', user='btsm', host='localhost')
conn.autocommit = True
cur = conn.cursor()

# Create a dedicated database for this project
cur.execute("SELECT 1 FROM pg_database WHERE datname='gebeya_slack'")
if not cur.fetchone():
    cur.execute('CREATE DATABASE gebeya_slack')
    print('Created database: gebeya_slack')
else:
    print('Database gebeya_slack already exists')

conn.close()

# Reconnect to the new database
conn = psycopg2.connect(dbname='gebeya_slack', user='btsm', host='localhost')
cur = conn.cursor()
print('Connected to gebeya_slack database')

Created database: gebeya_slack
Connected to gebeya_slack database


In [6]:
# Create tables
cur.execute('DROP TABLE IF EXISTS daily_sentiment')
cur.execute('DROP TABLE IF EXISTS message_features')
cur.execute('DROP TABLE IF EXISTS user_features')

cur.execute('''
    CREATE TABLE user_features (
        id              SERIAL PRIMARY KEY,
        user_id         TEXT NOT NULL,
        channel         TEXT NOT NULL,
        message_count   INTEGER DEFAULT 0,
        reply_count     INTEGER DEFAULT 0,
        reaction_count  INTEGER DEFAULT 0,
        mention_count   INTEGER DEFAULT 0,
        dominant_type   TEXT,
        created_at      TIMESTAMP DEFAULT NOW()
    )
''')

cur.execute('''
    CREATE TABLE message_features (
        id              SERIAL PRIMARY KEY,
        message_id      TEXT NOT NULL,
        channel         TEXT NOT NULL,
        user_id         TEXT,
        text_length     INTEGER DEFAULT 0,
        reply_count     INTEGER DEFAULT 0,
        reaction_count  INTEGER DEFAULT 0,
        message_type    TEXT,
        sentiment_score FLOAT,
        hour_of_day     INTEGER,
        day_of_week     INTEGER,
        created_at      TIMESTAMP DEFAULT NOW()
    )
''')

cur.execute('''
    CREATE TABLE daily_sentiment (
        id                SERIAL PRIMARY KEY,
        date              DATE NOT NULL,
        days_since_start  INTEGER NOT NULL,
        sentiment_score   FLOAT,
        message_count     INTEGER DEFAULT 0,
        created_at        TIMESTAMP DEFAULT NOW()
    )
''')

conn.commit()
print('Tables created: user_features, message_features, daily_sentiment')

Tables created: user_features, message_features, daily_sentiment


In [7]:
# Prepare message features
from textblob import TextBlob

def classify_message(text):
    if not isinstance(text, str) or text.strip() == '':
        return 'Other'
    t = text.lower()
    tech_kw = ['error','code','function','python','git','sql','model','pandas','docker']
    is_tech = any(kw in t for kw in tech_kw)
    if re.search(r'\?|\b(how|what|why|when|can|does|is|are)\b', t):
        return 'Question-Technical' if is_tech else 'Question-NonTechnical'
    if re.search(r'^(yes|no|sure|correct|exactly)\b', t):
        return 'Answer'
    return 'Comment-Technical' if is_tech else 'Comment-NonTechnical'

df['message_type'] = df['text'].apply(classify_message)
df['sentiment'] = df['text'].apply(
    lambda t: TextBlob(str(t)).sentiment.polarity if isinstance(t, str) else 0
)
df['text_length'] = df['text'].str.len().fillna(0).astype(int)
df['reply_count_clean'] = df['reply_count'].fillna(0).astype(int) if 'reply_count' in df.columns else 0
df['reaction_count_clean'] = df['reactions'].apply(
    lambda r: sum(x.get('count', 1) for x in r) if isinstance(r, list) else 0
)

print(f'Features prepared for {len(df):,} messages')

Features prepared for 19,602 messages


In [8]:
# Load message_features into PostgreSQL
message_rows = [
    (
        str(row['ts']),
        row['channel'],
        str(row.get('user', '')),
        int(row['text_length']),
        int(row['reply_count_clean']),
        int(row['reaction_count_clean']),
        row['message_type'],
        float(row['sentiment']),
        int(row['hour']) if pd.notna(row.get('hour')) else None,
        int(row['datetime'].weekday()) if pd.notna(row.get('datetime')) else None,
    )
    for _, row in df.iterrows()
]

execute_values(cur, '''
    INSERT INTO message_features
        (message_id, channel, user_id, text_length, reply_count,
         reaction_count, message_type, sentiment_score, hour_of_day, day_of_week)
    VALUES %s
''', message_rows)

conn.commit()
print(f'Inserted {len(message_rows):,} rows into message_features')

Inserted 19,602 rows into message_features


In [9]:
# Load user_features into PostgreSQL
user_agg = df.groupby(['user', 'channel']).agg(
    message_count=('ts', 'count'),
    reply_count=('reply_count_clean', 'sum'),
    reaction_count=('reaction_count_clean', 'sum'),
    dominant_type=('message_type', lambda x: x.value_counts().index[0])
).reset_index()

user_rows = [
    (
        str(row['user']),
        row['channel'],
        int(row['message_count']),
        int(row['reply_count']),
        int(row['reaction_count']),
        0,
        row['dominant_type'],
    )
    for _, row in user_agg.iterrows()
]

execute_values(cur, '''
    INSERT INTO user_features
        (user_id, channel, message_count, reply_count,
         reaction_count, mention_count, dominant_type)
    VALUES %s
''', user_rows)

conn.commit()
print(f'Inserted {len(user_rows):,} rows into user_features')

Inserted 1,205 rows into user_features


In [10]:
# Load daily_sentiment into PostgreSQL
training_start = df['datetime'].min().normalize()
df['days_since_start'] = (df['datetime'].dt.normalize() - training_start).dt.days

daily = df.groupby('days_since_start').agg(
    sentiment_score=('sentiment', 'mean'),
    message_count=('ts', 'count'),
    date=('datetime', lambda x: x.min().date())
).reset_index()

daily_rows = [
    (
        str(row['date']),
        int(row['days_since_start']),
        float(row['sentiment_score']),
        int(row['message_count']),
    )
    for _, row in daily.iterrows()
]

execute_values(cur, '''
    INSERT INTO daily_sentiment
        (date, days_since_start, sentiment_score, message_count)
    VALUES %s
''', daily_rows)

conn.commit()
print(f'Inserted {len(daily_rows):,} rows into daily_sentiment')

Inserted 104 rows into daily_sentiment


In [11]:
# Verify — run sample SQL queries
print('=== Sample SQL Queries ===')

cur.execute('SELECT COUNT(*) FROM message_features')
print(f'message_features rows: {cur.fetchone()[0]:,}')

cur.execute('SELECT COUNT(*) FROM user_features')
print(f'user_features rows:    {cur.fetchone()[0]:,}')

cur.execute('SELECT COUNT(*) FROM daily_sentiment')
print(f'daily_sentiment rows:  {cur.fetchone()[0]:,}')

print()
print('--- Top 5 most active users (all channels) ---')
cur.execute('''
    SELECT user_id, SUM(message_count) as total
    FROM user_features
    GROUP BY user_id
    ORDER BY total DESC
    LIMIT 5
''')
for row in cur.fetchall():
    print(f'  user={row[0][:10]}...  messages={row[1]}')

print()
print('--- Average sentiment per channel (top 5) ---')
cur.execute('''
    SELECT channel, ROUND(AVG(sentiment_score)::numeric, 3) as avg_sentiment
    FROM message_features
    GROUP BY channel
    ORDER BY avg_sentiment DESC
    LIMIT 5
''')
for row in cur.fetchall():
    print(f'  {row[0]}: {row[1]}')

cur.close()
conn.close()
client.close()
print('\nAll connections closed.')

=== Sample SQL Queries ===
message_features rows: 19,602
user_features rows:    1,205
daily_sentiment rows:  104

--- Top 5 most active users (all channels) ---
  user=U03V1AM5TF...  messages=1498
  user=U03UUR571A...  messages=1186
  user=U03UVHCV6K...  messages=1141
  user=U03UG32J3P...  messages=1031
  user=U03V6HMRPG...  messages=893

--- Average sentiment per channel (top 5) ---
  all-career-exercises: 0.125
  study-group: 0.118
  chang-w11: 0.116
  all-ideas: 0.107
  ab_test-group: 0.106

All connections closed.
